# Data Preprocess - Identify 20 missing MSOAs only in 2021 migration records

Given our analysis unit is 2011 MSOAs, we want to compare migration structures between 2011 and 2021, so both years should be expressed on the same spatial framework. 

The reason of using 2011 MSOAs is that everything revolves around 2011 boundaries:
- 2011 migration flows are reported on 2011 MSOAs.
- 2010 IMD baseline is aggregated 2011 MSOA with closer temporal match.
- Deprivation hierarchy is defined on 2011 MSOAs.
- The objective of this study is temporal comparison.

The aim for this file to help conver 2021 flows back to 2011 geography.

## Part 1 - Identify the types of affected London MSOAs among 20 missing from 2021 records.

There are 4 types of in the lookup:
- U = unchanged (have been filtered in previous data preprocessing file)
- **S = split** (multiple 2021 MSOAs are children, having a parent MSOA in 2011)
- **M = merge** (a single 2021 MSOA has multiple children MSOAs in 2011)
    - Hard to deal with;
    - Might use population-weighted splitting method
- X = complex

After this identification, we can convert every 2021 OD flow back with 2011 MSOA geography reference, and aggregate duplicated pairs for the split category.

**But we have not decided how to deal with the merge cases.**

In [1]:
# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION — update paths to match your local setup
# ══════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date
from scipy import stats
from pyprojroot import here
ROOT = here()

DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

# Input files (same as main preprocessing)
imd_2010_path       = DATA_DIR / 'imd_2010.xls'
census_od_2021_path = DATA_DIR / 'census_od_2021_msoa.csv'
census_od_2011_path = DATA_DIR / 'census_od_2011_oa.csv'
lookup_path         = DATA_DIR / 'NSPCL_NOV22_UK_LU.csv'
msoa_lookup_path    = DATA_DIR / 'msoa_2011_to_2021_lookup.csv'
msoa_lookup_identification_path    = DATA_DIR / 'MSOA_2011_to_2021_lookup_for_identification.csv'

# NEW: all-England KS101 population file
ks101_allengland_path = DATA_DIR / 'ks101ew_lsoa_2011_allengland.csv'

# Synthetic code for all non-London areas
EXTERNAL_CODE = 'EXT_OUTSIDE'

# IMD 2010 column names
IMD_2010_LSOA_COL  = 'LSOA CODE'
IMD_2010_SCORE_COL = 'IMD SCORE'

---

## 0. Loading data

In [2]:
# ---- IMD 2010 (LSOA level, all England) ----
IMD_2010_LSOA_COL  = 'LSOA CODE'
IMD_2010_SCORE_COL = 'IMD SCORE'

imd_2010 = pd.read_excel(imd_2010_path, sheet_name='IMD 2010')
imd_2010.columns = imd_2010.columns.str.strip()
print(f'IMD 2010: {len(imd_2010)} LSOAs')


IMD 2010: 32482 LSOAs


In [3]:
# ---- NSPCL postcode lookup (LSOA → MSOA mapping) ----
nspcl = pd.read_csv(lookup_path, encoding='ISO-8859-1', low_memory=False)
lsoa_to_msoa = nspcl[['lsoa11cd', 'msoa11cd']].drop_duplicates().dropna()

In [4]:
# ---- KS101EW population (all England & Wales LSOAs) ----
ks101 = pd.read_csv(ks101_allengland_path)
# Detect LSOA code column: find column where values match E01/W01 pattern
_ks_code_col = next(
    c for c in ks101.columns
    if ks101[c].astype(str).str.match(r'^[EW]01\d{6}$').mean() > 0.9
)
_ks_pop_col = next(c for c in ks101.columns if 'all usual residents' in c.lower())
lsoa_pop = (ks101[[_ks_code_col, _ks_pop_col]]
            .rename(columns={_ks_code_col: 'lsoa11cd', _ks_pop_col: 'pop'}))
lsoa_pop['pop'] = pd.to_numeric(lsoa_pop['pop'], errors='coerce').fillna(0)
print(f'KS101EW: {len(lsoa_pop)} LSOAs, total pop = {lsoa_pop["pop"].sum():,.0f}')

KS101EW: 34753 LSOAs, total pop = 56,075,912


In [5]:
# ---- 2011 to 2021 lookup table for identification ----
lookup_identification = pd.read_csv(msoa_lookup_identification_path)

print(lookup_identification.columns.tolist())

['MSOA11CD', 'MSOA11NM', 'CHNGIND', 'MSOA21CD', 'MSOA21NM', 'LAD22CD', 'LAD22NM', 'LAD22NMW', 'ObjectId']


---

## 1. Counting MSOA change types from 2011 to 2021

In [6]:
## count change types

lookup_identification['CHNGIND'].value_counts(dropna=False)

CHNGIND
U    7080
S     159
M      38
X       9
Name: count, dtype: int64

> **Is it valid to use London only identification? Given we now consider external flows already.**
>
> I think yes, becuase we only focus on London nodes, and we aggregate all outside London MSOAs as a single one.

In [7]:
# ---- Lookups ----
lookup = pd.read_csv(lookup_path, encoding='ISO-8859-1', low_memory=False)
msoa_11_21 = pd.read_csv(msoa_lookup_path)
msoa_11_21.columns = msoa_11_21.columns.str.strip().str.lower()

---

## 2. Aggregate IMD 2010 from LSOA to MSOA

In [8]:
print(imd_2010.columns.tolist())

['LSOA CODE', 'LA CODE', 'LA NAME', 'GOR CODE', 'GOR NAME', 'IMD SCORE', 'RANK OF IMD SCORE (where 1 is most deprived)']


In [9]:
# ---- Join IMD scores → MSOA geography → population weights ----
imd_lsoa = imd_2010[[IMD_2010_LSOA_COL, IMD_2010_SCORE_COL]].rename(
    columns={IMD_2010_LSOA_COL: 'lsoa11cd', IMD_2010_SCORE_COL: 'imd_score'})

# LSOA → MSOA
imd_with_msoa = pd.merge(imd_lsoa, lsoa_to_msoa, on='lsoa11cd', how='inner')
print(f'IMD LSOAs matched to MSOA: {len(imd_with_msoa)} / {len(imd_lsoa)}')

# Add population weights
imd_with_msoa = pd.merge(imd_with_msoa, lsoa_pop, on='lsoa11cd', how='left')
imd_with_msoa['pop'] = imd_with_msoa['pop'].fillna(0)

IMD LSOAs matched to MSOA: 31672 / 32482


In [10]:
# ---- London scope ----
london_boroughs = [
    'City of London', 'Barking and Dagenham', 'Barnet', 'Bexley', 'Brent',
    'Bromley', 'Camden', 'Croydon', 'Ealing', 'Enfield', 'Greenwich',
    'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering',
    'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea',
    'Kingston upon Thames', 'Lambeth', 'Lewisham', 'Merton', 'Newham',
    'Redbridge', 'Richmond upon Thames', 'Southwark', 'Sutton',
    'Tower Hamlets', 'Waltham Forest', 'Wandsworth', 'Westminster'
]
london_lookup = (lookup[lookup['ladnm'].isin(london_boroughs)]
                 [['lsoa11cd', 'msoa11cd', 'ladnm']].drop_duplicates())
london_msoas = set(london_lookup['msoa11cd'].unique())
print(f'London MSOAs (geography): {len(london_msoas)}')

London MSOAs (geography): 983


In [11]:
# ---- Population-weighted mean per MSOA ----
# (Same fallback as main notebook: if all weights are zero, use simple mean)
def pop_weighted_mean(group):
    total_pop = group['pop'].sum()
    if total_pop > 0:
        return np.average(group['imd_score'], weights=group['pop'])
    else:
        return group['imd_score'].mean()

msoa_imd = (imd_with_msoa
            .groupby('msoa11cd')
            .apply(pop_weighted_mean, include_groups=False)
            .reset_index(name='IMD_2010_national'))

print(f'MSOAs with aggregated IMD: {len(msoa_imd)}')
print(f'  London MSOAs:     {msoa_imd["msoa11cd"].isin(london_msoas).sum()}')
print(f'  Non-London MSOAs: {(~msoa_imd["msoa11cd"].isin(london_msoas)).sum()}')


MSOAs with aggregated IMD: 6778
  London MSOAs:     983
  Non-London MSOAs: 5795


In [ ]:
# ---- Assign national wealth deciles ----
# qcut splits into 10 bins by score. Higher IMD score = more deprived.
# pd.qcut with labels=False gives bin 0 = lowest scores = LEAST deprived.
# We want Decile 1 = most deprived, so: Decile = 11 - (bin + 1)
msoa_imd['Wealth_Decile_National'] = (
    11 - (pd.qcut(msoa_imd['IMD_2010_national'], 10, labels=False) + 1)
)

# Verify direction: D1 should have highest IMD scores (most deprived)
d1_score = msoa_imd.loc[msoa_imd['Wealth_Decile_National'] == 1,
                         'IMD_2010_national'].mean()
d10_score = msoa_imd.loc[msoa_imd['Wealth_Decile_National'] == 10,
                          'IMD_2010_national'].mean()
assert d1_score > d10_score, \
    f'Decile direction wrong: D1 mean={d1_score:.1f}, D10 mean={d10_score:.1f}'
print(f'\nDecile direction verified: D1 (most deprived, mean={d1_score:.1f}) > '
      f'D10 (least deprived, mean={d10_score:.1f})')


Decile direction verified: D1 (most deprived, mean=50.7) > D10 (least deprived, mean=5.7)


In [13]:
# ---- Summary: national decile composition ----
print(f'\nNational decile distribution:')
print(f'{"Decile":>6s} {"N_total":>8s} {"N_London":>9s} {"Mean IMD Score":>9s}')
for d in range(1, 11):
    mask = msoa_imd['Wealth_Decile_National'] == d
    n_total = mask.sum()
    n_london = (mask & msoa_imd['msoa11cd'].isin(london_msoas)).sum()
    mean_imd = msoa_imd.loc[mask, 'IMD_2010_national'].mean()
    print(f'  D{d:>2d}   {n_total:>7d}  {n_london:>8d}  {mean_imd:>9.2f}')


National decile distribution:
Decile  N_total  N_London Mean IMD Score
  D 1       678       107      50.73
  D 2       678       182      36.99
  D 3       678       149      29.63
  D 4       677       135      24.06
  D 5       678       107      19.72
  D 6       678        76      16.22
  D 7       677        74      13.41
  D 8       678        65      11.03
  D 9       678        52       8.74
  D10       678        36       5.73


---

## 3. Restrict to london only

In [15]:
## Restrict to London only
lookup_london = lookup_identification[
    lookup_identification['MSOA11CD'].isin(london_msoas)
].copy()

lookup_london['CHNGIND'].value_counts()

CHNGIND
U    963
S     38
M      2
Name: count, dtype: int64

---

## 4. Find all non-unchanges London MSOAs

In [17]:
changed_london = lookup_london[
    lookup_london['CHNGIND'] != 'U'
].copy()

print(changed_london.shape)

changed_london[
    ['MSOA11CD',
     'MSOA11NM',
     'MSOA21CD',
     'MSOA21NM',
     'CHNGIND']
].sort_values('MSOA11CD')

changed_london.head(20)

(40, 9)


,MSOA11CD,MSOA11NM,CHNGIND,MSOA21CD,MSOA21NM,LAD22CD,LAD22NM,LAD22NMW,ObjectId
6481,E02000664,Lewisham 012,S,E02007008,Lewisham 040,E09000023,Lewisham,NaN,6482
6488,E02000664,Lewisham 012,S,E02007009,Lewisham 041,E09000023,Lewisham,NaN,6489
6565,E02000257,Ealing 020,S,E02006968,Ealing 041,E09000009,Ealing,NaN,6566
6570,E02000769,Redbridge 019,S,E02007017,Redbridge 037,E09000026,Redbridge,NaN,6571
6571,E02000257,Ealing 020,S,E02006969,Ealing 042,E09000009,Ealing,NaN,6572
6575,E02000274,Ealing 037,S,E02006970,Ealing 043,E09000009,Ealing,NaN,6576
6577,E02000274,Ealing 037,S,E02006971,Ealing 044,E09000009,Ealing,NaN,6578
6578,E02000769,Redbridge 019,S,E02007018,Redbridge 038,E09000026,Redbridge,NaN,6579
6580,E02000780,Redbridge 030,S,E02007019,Redbridge 039,E09000026,Redbridge,NaN,6581
6581,E02000528,Hounslow 003,S,E02006972,Hounslow 030,E09000018,Hounslow,NaN,6582


---

## 5. Count split parents

In [18]:
split_msoas = lookup_london[
    lookup_london['CHNGIND'] == 'S'
]

print(split_msoas['MSOA11CD'].nunique())

split_msoas[
    ['MSOA11CD',
     'MSOA11NM',
     'MSOA21CD',
     'MSOA21NM']
].sort_values('MSOA11CD')

18


,MSOA11CD,MSOA11NM,MSOA21CD,MSOA21NM
6790,E02000053,Barnet 030,E02006953,Barnet 042
6794,E02000053,Barnet 030,E02006954,Barnet 043
6796,E02000109,Brent 017,E02006955,Brent 035
6798,E02000109,Brent 017,E02006956,Brent 036
6667,E02000220,Croydon 027,E02006987,Croydon 046
6669,E02000220,Croydon 027,E02006988,Croydon 047
6565,E02000257,Ealing 020,E02006968,Ealing 041
6571,E02000257,Ealing 020,E02006969,Ealing 042
6575,E02000274,Ealing 037,E02006970,Ealing 043
6577,E02000274,Ealing 037,E02006971,Ealing 044


---

## 6. Count merge parents

In [19]:
merge_msoas = lookup_london[
    lookup_london['CHNGIND'] == 'M'
]

print(merge_msoas['MSOA11CD'].nunique())

merge_msoas[
    ['MSOA11CD',
     'MSOA11NM',
     'MSOA21CD',
     'MSOA21NM']
].sort_values('MSOA21CD')

2


,MSOA11CD,MSOA11NM,MSOA21CD,MSOA21NM
6824,E02000189,Camden 024,E02007115,Camden 029
6825,E02000190,Camden 025,E02007115,Camden 029


---

## 7. Examine 20 missing MSOAs

copying missing MSOA resuls from EDA_4_2nd_draft_refine file in §0b.

In [20]:
missing_msoas = ['E02000053', 'E02000109', 'E02000189', 'E02000190', 'E02000220', 'E02000257', 'E02000274', 'E02000316', 'E02000365', 'E02000371', 'E02000528', 'E02000664', 'E02000726', 'E02000750', 'E02000769', 'E02000780', 'E02000809', 'E02000891', 'E02000924', 'E02006929']

In [21]:
missing_lookup = lookup_london[
    lookup_london['MSOA11CD'].isin(missing_msoas)
]

missing_lookup[
    ['MSOA11CD',
     'MSOA11NM',
     'MSOA21CD',
     'MSOA21NM',
     'CHNGIND']
].sort_values('MSOA11CD')

,MSOA11CD,MSOA11NM,MSOA21CD,MSOA21NM,CHNGIND
6790,E02000053,Barnet 030,E02006953,Barnet 042,S
6794,E02000053,Barnet 030,E02006954,Barnet 043,S
6796,E02000109,Brent 017,E02006955,Brent 035,S
6798,E02000109,Brent 017,E02006956,Brent 036,S
6824,E02000189,Camden 024,E02007115,Camden 029,M
6825,E02000190,Camden 025,E02007115,Camden 029,M
6667,E02000220,Croydon 027,E02006987,Croydon 046,S
6669,E02000220,Croydon 027,E02006988,Croydon 047,S
6565,E02000257,Ealing 020,E02006968,Ealing 041,S
6571,E02000257,Ealing 020,E02006969,Ealing 042,S


In [22]:
missing_lookup['CHNGIND'].value_counts()

CHNGIND
S    38
M     2
Name: count, dtype: int64

In [24]:
lookup_london['CHNGIND'].value_counts()

CHNGIND
U    963
S     38
M      2
Name: count, dtype: int64

Among 20 missing MSOAs, we have 38 rows split from 2011 to 2021, and 2 merge. We're safe there is no complex cases.

But I'm still struggling with how to identify those missing MSOAs, and how to convert those flows in 2011 and add back to the whole data for analysis.

Another question is since we consider external flows from all England and Wales, should we care such MSOA coding for them? 

---

## 8. Checking lookup results - missing MSOAs and the whole lookup.

Likely reasons are:
1. counting unmatched after transformation
2. split MSOAs generate multiple outputs, but have partial matches
3. mix MSOA-level vs row-level logic

In [25]:
missing_lookup.groupby('CHNGIND')['MSOA11CD'].nunique()

CHNGIND
M     2
S    18
Name: MSOA11CD, dtype: int64

In [26]:
missing_lookup['MSOA11CD'].nunique()

20

In [27]:
missing_lookup.groupby('MSOA11CD')['MSOA21CD'].nunique().sort_values(ascending=False)

MSOA11CD
E02000891    3
E02000726    3
E02000053    2
E02000109    2
E02000924    2
E02000809    2
E02000780    2
E02000769    2
E02000750    2
E02000664    2
E02000528    2
E02000371    2
E02000365    2
E02000316    2
E02000274    2
E02000257    2
E02000220    2
E02006929    2
E02000190    1
E02000189    1
Name: MSOA21CD, dtype: int64

In [29]:
set_missing = set(missing_msoas)
set_split_merge = set(lookup[lookup['CHNGIND'].isin(['S','M'])]['MSOA11CD'])

intersection = set_missing & set_split_merge

len(intersection), intersection

KeyError: 'CHNGIND'